# 🔎 UNSW-NB15 KNN Similarity Intelligence & Pattern Retrieval Engine
### Architecture: KNN (Similarity Intelligence) + XGBoost (Primary Classifier)

This notebook trains the **StandardScaler + KNN (NearestNeighbors / KD-Tree)** engine on the **UNSW-NB15** dataset.

**Flow:**
```
New network flow ──▶ StandardScaler ──▶ KNN (k=5) ──▶ Find nearest historical flows ──▶ "Similar to previous DoS traffic"
```

**Output Schema:**
```json
{
  "prediction": "DoS",
  "similarity_confidence": 0.96,
  "nearest_examples": 5
}
```

### 1. Install Dependencies & Import Libraries

In [ ]:
!pip install kagglehub scikit-learn pandas numpy joblib matplotlib seaborn -q

import os
import glob
import json
import numpy as np
import pandas as pd
import joblib
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

print('Libraries loaded successfully!')

### 2. Download Dataset via KaggleHub & Load CSVs

In [ ]:
print('Downloading official UNSW-NB15 dataset via KaggleHub...')
dataset_path = kagglehub.dataset_download('mrwellsdavid/unsw-nb15')

print(f'[SUCCESS] Dataset downloaded to: {dataset_path}')
all_files = os.listdir(dataset_path)
print('Files in dataset:', all_files)

# Locate training and testing CSVs
train_file = next(os.path.join(dataset_path, f) for f in all_files if 'training' in f.lower() or 'train' in f.lower())
test_file = next(os.path.join(dataset_path, f) for f in all_files if 'testing' in f.lower() or 'test' in f.lower())

df_train = pd.read_csv(train_file)
df_test = pd.read_csv(test_file)

print(f'\nTrain Shape: {df_train.shape} | Test Shape: {df_test.shape}')
df_train.head(2)

### 3. Data Cleaning, Preprocessing & Encoders

In [ ]:
# 1. Clean IDs and standardize attack category labels
for df in [df_train, df_test]:
    if 'id' in df.columns:
        df.drop(columns=['id'], inplace=True)
    if 'attack_cat' in df.columns:
        df['attack_cat'] = df['attack_cat'].fillna('Normal').astype(str).str.strip()

# 2. Separate Features & Targets
drop_cols = ['label', 'attack_cat']
feature_cols = [c for c in df_train.columns if c not in drop_cols]

X_train_raw = df_train[feature_cols].copy()
y_train_label = df_train['label'].copy()
y_train_cat = df_train['attack_cat'].copy()

X_test_raw = df_test[feature_cols].copy()
y_test_label = df_test['label'].copy()
y_test_cat = df_test['attack_cat'].copy()

# 3. Identify and Encode Categorical Features (proto, service, state)
cat_cols = X_train_raw.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical features detected: {cat_cols}')

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train_raw[col].astype(str), X_test_raw[col].astype(str)], axis=0)
    le.fit(combined)
    X_train_raw[col] = le.transform(X_train_raw[col].astype(str))
    X_test_raw[col] = le.transform(X_test_raw[col].astype(str))
    encoders[col] = le

# 4. Convert all feature columns to float32 and handle infinite/NaN values
X_train_raw = X_train_raw.astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0)
X_test_raw = X_test_raw.astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0)

print(f'\n[OK] Preprocessed {len(feature_cols)} features.')
print('\nAttack Category Distribution (Train):')
print(y_train_cat.value_counts())

### 4. Build Diverse Reference Memory & Fit StandardScaler
Extracts balanced flow exemplars across all attack families for instantaneous KD-Tree query execution.

In [ ]:
np.random.seed(42)
SAMPLE_SIZE = 15000

stratified_indices = []
cats = df_train['attack_cat'].unique()
samples_per_cat = max(350, SAMPLE_SIZE // len(cats))

for cat in cats:
    idx = df_train[df_train['attack_cat'] == cat].index
    if len(idx) > samples_per_cat:
        sampled = np.random.choice(idx, size=samples_per_cat, replace=False)
    else:
        sampled = idx.values
    stratified_indices.extend(sampled)

stratified_indices = list(set(stratified_indices))
print(f'Curated {len(stratified_indices)} high-density historical flow exemplars.')

X_ref = X_train_raw.iloc[stratified_indices].reset_index(drop=True)
y_ref_cat = y_train_cat.iloc[stratified_indices].reset_index(drop=True)
y_ref_label = y_train_label.iloc[stratified_indices].reset_index(drop=True)

ref_metadata = df_train.iloc[stratified_indices][['proto', 'service', 'state', 'dur', 'sbytes', 'dbytes', 'attack_cat', 'label']].copy().reset_index(drop=True)
for c in ['proto', 'service', 'state', 'attack_cat']:
    ref_metadata[c] = ref_metadata[c].astype(str)

# Fit StandardScaler on reference vectors
print('Fitting StandardScaler...')
scaler = StandardScaler()
X_ref_scaled = scaler.fit_transform(X_ref)
print('StandardScaler fit completed successfully.')

### 5. Fit NearestNeighbors (KD-Tree) Engine & Companion Classifier

In [ ]:
K_NEIGHBORS = 5
print(f'Fitting NearestNeighbors index (k={K_NEIGHBORS}, algorithm=kd_tree)...')
knn_model = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric='euclidean', algorithm='kd_tree')
knn_model.fit(X_ref_scaled)

knn_clf = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, metric='euclidean', weights='distance')
knn_clf.fit(X_ref_scaled, y_ref_cat)
print('KNN Engine successfully fitted.')

### 6. Evaluate Pattern Matching Accuracy on Unseen Test Traffic

In [ ]:
test_eval_idx = np.random.choice(len(df_test), size=min(5000, len(df_test)), replace=False)
X_test_subset = X_test_raw.iloc[test_eval_idx]
y_test_cat_subset = y_test_cat.iloc[test_eval_idx]

X_test_scaled = scaler.transform(X_test_subset)
y_pred_cat = knn_clf.predict(X_test_scaled)

acc = accuracy_score(y_test_cat_subset, y_pred_cat)
print(f'KNN Pattern Matching Accuracy: {acc * 100:.2f}%\n')
print(classification_report(y_test_cat_subset, y_pred_cat, zero_division=0))

### 7. Test Inference Pipeline & Verify CyberGuard UI Output Schema

In [ ]:
def query_knn_similarity(flow_dict, k=5):
    row = {}
    for feat in feature_cols:
        val = flow_dict.get(feat, 0)
        if feat in encoders:
            le = encoders[feat]
            val_str = str(val)
            val = le.transform([val_str])[0] if val_str in le.classes_ else 0
        row[feat] = float(val)
    
    df_in = pd.DataFrame([row])[feature_cols].astype(np.float32)
    scaled_in = scaler.transform(df_in)
    distances, indices = knn_model.kneighbors(scaled_in, n_neighbors=k)
    
    nearest_rows = ref_metadata.iloc[indices[0]]
    avg_dist = float(np.mean(distances[0]))
    similarity_conf = round(float(1.0 / (1.0 + (avg_dist / 2.0))), 2)
    dominant_cat = str(nearest_rows['attack_cat'].mode()[0])
    
    return {
        'prediction': dominant_cat,
        'similarity_confidence': similarity_conf,
        'nearest_examples': k,
        'explanation': f'Similar to previous {dominant_cat} traffic signatures',
        'nearest_historical_flows': nearest_rows[['attack_cat', 'proto', 'service', 'dur', 'sbytes', 'dbytes']].to_dict(orient='records')
    }

# Test with DoS traffic sample
sample_dos = df_test[df_test['attack_cat'] == 'DoS'].iloc[0].to_dict()
res = query_knn_similarity(sample_dos, k=5)
print('=== INFERENCE OUTPUT FOR CYBERGUARD UI ===')
print(json.dumps(res, indent=2))

### 8. Export Model Artifacts (.joblib)

In [ ]:
os.makedirs('artifacts', exist_ok=True)
knn_path = os.path.join('artifacts', 'knn_unsw_nb15.joblib')
scaler_path = os.path.join('artifacts', 'knn_scaler.joblib')
ref_data_path = os.path.join('artifacts', 'knn_reference_samples.joblib')

payload = {
    'features': feature_cols,
    'metadata': ref_metadata,
    'encoders': encoders
}

joblib.dump(knn_model, knn_path)
joblib.dump(scaler, scaler_path)
joblib.dump(payload, ref_data_path)

print(f'[OK] KNN Model saved to: {knn_path}')
print(f'[OK] Scaler saved to: {scaler_path}')
print(f'[OK] Reference library saved to: {ref_data_path}')

# Trigger direct browser download in Google Colab
try:
    from google.colab import files
    files.download(knn_path)
    files.download(scaler_path)
    files.download(ref_data_path)
except Exception:
    pass